# 🧬 NeuralAI Mamba K2 — SFT Training

**Model**: `state-spaces/mamba-790m-hf` (790M parameters, Mamba SSM architecture)
**Data**: 10K–15K UltraChat conversational samples
**Steps**: 500–1000 (SFT)
**Goal**: Loss < 3.0 → Merged model → Q4_K_M GGUF

Built by De'Andrew Harris & Gemini | NeuralAI

In [ ]:
# @title ⚙️ Install Dependencies (1 min)
!pip install -q torch transformers datasets accelerate peft mamba-ssm
!pip install -q bitsandbytes sentencepiece  # for optional quantization
print('✅ Dependencies installed')

In [ ]:
# @title 📦 Load Base Model
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM, TrainingArguments, Trainer
from datasets import load_dataset
from peft import LoraConfig, get_peft_model, TaskType

BASE_MODEL = "state-spaces/mamba-790m-hf"
OUTPUT_DIR = "./mamba-k2-sft"

print(f'Loading tokenizer: {BASE_MODEL}')
tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL)
tokenizer.pad_token = tokenizer.eos_token

print(f'Loading base model (fp16)...')
model = AutoModelForCausalLM.from_pretrained(
    BASE_MODEL,
    torch_dtype=torch.float16,
    device_map="auto",
    trust_remote_code=True
)
print(f'Base model loaded: {sum(p.numel() for p in model.parameters()):,} params')

In [ ]:
# @title 🎯 Apply LoRA (rank 16, targeting Mamba projections)
lora_config = LoraConfig(
    r=16,
    lora_alpha=32,
    target_modules=["in_proj", "dt_proj", "x_proj", "out_proj"],
    lora_dropout=0.05,
    bias="none",
    task_type=TaskType.CAUSAL_LM,
)

model = get_peft_model(model, lora_config)
trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
total = sum(p.numel() for p in model.parameters())
print(f'Trainable: {trainable:,} / {total:,} ({100*trainable/total:.1f}%)')
model.print_trainable_parameters()

In [ ]:
# @title 📊 Load & Tokenize UltraChat (10K-15K samples)
print('Loading UltraChat dataset (15K samples)...')
dataset = load_dataset("stingning/ultrachat", split="train[:15000]")
print(f'Loaded: {len(dataset)} conversations')

def format_mamba_chat(examples):
    """Format UltraChat conversations into Mamba K2 instruction format."""
    texts = []
    for messages in examples["data"]:
        # UltraChat 'data' is a list of message dicts
        if isinstance(messages, list):
            formatted = []
            for msg in messages:
                if isinstance(msg, dict):
                    content = msg.get("content", "")
                    role = msg.get("role", "user")
                    if role == "user":
                        formatted.append(f"<|user|>\n{content}")
                    elif role == "assistant":
                        formatted.append(f"<|assistant|>\n{content}")
            if formatted:
                texts.append("\n".join(formatted) + "\n<|endoftext|>")
    return texts

# Tokenize
MAX_LENGTH = 1024

def tokenize_fn(examples):
    texts = format_mamba_chat(examples)
    if not texts:
        return {"input_ids": [], "attention_mask": [], "labels": []}
    result = tokenizer(
        texts,
        truncation=True,
        padding="max_length",
        max_length=MAX_LENGTH,
        return_tensors="np"
    )
    result["labels"] = result["input_ids"].copy()
    return result

print('Tokenizing...')
tokenized = dataset.map(
    tokenize_fn,
    batched=True,
    batch_size=100,
    remove_columns=dataset.column_names
)

# Split train/eval
split = tokenized.train_test_split(test_size=0.05, seed=42)
train_ds = split["train"]
eval_ds = split["test"]
print(f'Train: {len(train_ds)} | Eval: {len(eval_ds)}')

In [ ]:
# @title 🏋️ Train — 500-1000 SFT Steps
TRAIN_STEPS = 750  # Adjust: 500 for quick run, 1000 for thorough

training_args = TrainingArguments(
    output_dir=OUTPUT_DIR,
    num_train_epochs=3,
    per_device_train_batch_size=4,
    per_device_eval_batch_size=4,
    gradient_accumulation_steps=4,
    learning_rate=2e-4,
    warmup_steps=100,
    max_steps=TRAIN_STEPS,
    logging_steps=25,
    eval_steps=100,
    save_steps=250,
    save_total_limit=3,
    fp16=True,
    bf16=False,
    report_to="none",
    remove_unused_columns=False,
    dataloader_num_workers=2,
    load_best_model_at_end=True,
    metric_for_best_model="eval_loss",
    greater_is_better=False,
)

print(f'Training {TRAIN_STEPS} steps...')
print(f'Effective batch size: {training_args.per_device_train_batch_size * training_args.gradient_accumulation_steps}')

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_ds,
    eval_dataset=eval_ds,
    tokenizer=tokenizer,
)

trainer.train()

In [ ]:
# @title 🔬 Quick Evaluation — Check Generated Quality
model.eval()
test_prompts = [
    "<|user|>\nWhat is the capital of France?",
    "<|user|>\nExplain quantum entanglement in simple terms.",
    "<|user|>\nWrite a haiku about artificial intelligence.",
]

for prompt in test_prompts:
    inputs = tokenizer(prompt, return_tensors="pt").to(model.device)
    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=128,
            temperature=0.7,
            do_sample=True,
            top_p=0.9,
            pad_token_id=tokenizer.eos_token_id
        )
    result = tokenizer.decode(outputs[0], skip_special_tokens=False)
    print(f"\n{'='*60}")
    print(result)
    print(f"{'='*60}")

In [ ]:
# @title 💾 Merge LoRA → Full Model & Save
from peft import PeftModel
import shutil

print('Merging LoRA into base model...')
merged = model.merge_and_unload()

MERGE_DIR = "./mamba-k2-merged"
print(f'Saving merged model to {MERGE_DIR}...')
merged.save_pretrained(MERGE_DIR)
tokenizer.save_pretrained(MERGE_DIR)
print('✅ Merged model saved')

# Create a zip for download
shutil.make_archive("mamba-k2-merged", "zip", MERGE_DIR)
print('✅ Created mamba-k2-merged.zip for download')
print(f'\n📊 Training complete!')
print(f'  Final eval loss: {trainer.state.log_history[-1].get("eval_loss", "N/A") if hasattr(trainer, "state") else "N/A"}')
print(f'  Steps: {trainer.state.global_step if hasattr(trainer, "state") else "N/A"}')

In [ ]:
# @title 📦 (Optional) Convert to GGUF — Run Locally After Download
print('''
To convert the merged model to GGUF for LM Studio:

1. Download mamba-k2-merged.zip from Colab
2. Upload to HuggingFace Hub
3. Request GGUF conversion via mradermacher's converter:
   https://huggingface.co/mradermacher

Or, use the community Q4_K_M quantization at:
   mradermacher/mamba-790m-hf-GGUF
   → mamba-790m-hf.Q4_K_M.gguf (459 MB)

Rename to: NeuralAI-Mamba-K2-SFT-Q4_K_M.gguf
Load in LM Studio → Done!
''')

## 📈 Expected Results

| Metric | Target |
|--------|--------|
| Training steps | 500–1000 |
| SFT Loss | < 3.0 |
| Samples | 10K–15K UltraChat |
| Effective batch | 16 |
| LoRA rank | 16 |
| Trainable params | ~5-10M |

### Comparison to Mamba K1

| | K1 | K2 |
|---|---|---|
| Params | 130M | 790M (6×) |
| Data | 1K | 15K (15×) |
| Steps | 50 | 750 (15×) |
| K1 Loss | 6.78 | < 3.0 target |

**Next**: After successful training → `NeuralAI Mamba K3` on `mamba-2.8b-hf`